In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile as tiff
import seaborn as sns

import os
import sys
from pathlib import Path

project_root = Path.cwd().parent
# Add the root directory to Python's module search path
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import algorithms.database as database
import algorithms.analysis as analysis
from config import microns_per_pixel, main_image_path, processed_path, live_imaging_path,drugs_image_path
from pathlib import Path


In [ ]:
def distance(coord,centre_x,centre_y,ellipse_w,ellipse_h):
    a = ellipse_w/2
    b = ellipse_h/2
    #Calculate the radius of the ellipse that coord would lie on, given a centre point, and a and b.
    #Since the original radius is always 1 ((x-c1)**2/a**2 + (y-c2)**2/b**2=1), this tells us, in a radial sense, how "far" the point is from the centre of the embryo.
    new_r = (coord[0]-centre_x)**2/a**2 + (coord[1]-centre_y)**2/b**2
    return new_r

Displays properties of nodes as a function of their centrality: distance from the embryo centre

In [ ]:
calc_drugs = False
calc_live = False
show=True

nodes_data = pd.DataFrame(columns=["Stage","n","Row","Column","Weight Correlation","Degree Correlation"])
collected_data = []


#Some of this code could be put into common functions used by this file and also spatial_analysis_grid.ipynb
nodes_path = processed_path / "skeleton_networks"
for file_name in os.listdir(nodes_path):
        #Get embryo ID
        embryo_ID, end = file_name.split("_")
        embryo_ID = int(embryo_ID)
        if end=="nodes.csv":
            nodes = pd.read_csv(nodes_path / f"{embryo_ID}_nodes.csv", index_col = 0)
            adj = pd.read_csv(nodes_path / f"{embryo_ID}_adj.csv", index_col = 0)
            adj.columns = adj.columns.astype(int) #Convert headings to ints
            edges = pd.read_csv(nodes_path / f"{embryo_ID}_edges.csv", index_col = 0)

            #split into grid regions
            xmin = nodes["x"].quantile(0.01)
            xmax = nodes["x"].quantile(0.99)
            ymax = nodes["y"].quantile(0.99)

            metadata_df = database.initialise_metadata()
            row = metadata_df.loc[embryo_ID]
            #ymin is read from the anterior point
            ymin = row["Anterior_Y"]/microns_per_pixel

            ell_x,ell_y = row["Ellipse_X"]/microns_per_pixel, row["Ellipse_Y"]/microns_per_pixel
            ell_w,ell_h = row["Ellipse_W"]/microns_per_pixel, row["Ellipse_H"]/microns_per_pixel
            centre_x = ell_x
            centre_y = ell_x

            stage = int(row["Stage"])
            n = int(row["n"])
            condition = row["Condition"]
            
            if pd.isna(condition):
                image = cv2.imread(main_image_path / f'hh{stage}_n{n} plain_scaled.jpg')
            else:
                drug = row["Drug"]
                date = row["Experiment_Date"]
                if "live" in drug and calc_live:
                    print("drug")
                    image = cv2.imread(live_imaging_path / f"hh{stage}_n{n}" / f'hh{stage}_n{n}_{condition} plain_scaled.jpg')
                elif not "live" in drug and calc_drugs:
                    image = cv2.imread(drugs_image_path / f"{experiment_date}_{drug}" / f'hh{stage}_n{n}_{condition} plain_scaled.jpg')
                else:
                    continue

            height = len(image)
            width = len(image[0])

            #Store the distance to the centrepoint per node, in microns
           
            nodes["centre_dist"] = [distance([nodes.loc[i,"x"],nodes.loc[i,"y"]],ell_x,ell_y,ell_w,ell_h) for i in nodes.index]
            #Store node degree
            neighbours = adj.count()
            nodes["degree"] = neighbours

            fig, ax = plt.subplots(figsize=(10, 8))
            #plot iamge
            ax.imshow(image, cmap=plt.cm.gray,alpha=0.5)
            
            #plot nodes
            scatter = ax.scatter(nodes['x'], nodes['y'], c=nodes['centre_dist'], s=nodes['weight']*3, alpha=0.6, label='Nodes',cmap="winter")
            plt.colorbar(scatter, label="Distance from Embryo Centre (Ellipse)")
            scatter = ax.scatter(centre_x, centre_y, c="red", s=10, alpha=1, label='Embryo Centre',cmap="winter")

            plt.title(f"Blood Island Location, Stage: HH{stage}, n: {n}")
            plt.legend()
            plt.show()

            sns.scatterplot(data=nodes, x="centre_dist",y="degree")
            plt.xlabel('Distance from Embryo Centre')
            plt.ylabel('Node Degree')
            plt.title(f"Blood Island Degree and Distance from Embryo Centre, Stage: HH{stage}, n: {n}")
            plt.show()

            sns.scatterplot(data=nodes, x="centre_dist",y="weight")
            plt.xlabel('Distance from Embryo Centre')
            plt.ylabel('Node Weight')
            plt.title(f"Blood Island Size and Distance from Embryo Centre, Stage: HH{stage}, n: {n}")
            plt.show()


            collected_data.append({
                                    "Embryo_ID": embryo_ID,
                                    "Stage": stage,
                                    "Condition": condition,
                                    "n": n,
                                    "Weight-Location Correlation": np.corrcoef(nodes["weight"], nodes["centre_dist"]),
                                    "Degree-Location Correlation": np.corrcoef(nodes["degree"], nodes["centre_dist"]),
                                })
nodes_data=pd.DataFrame(collected_data)
             